# LAB 10 — Tokenizer, Offset Mapping & Span Alignment (Extractive QA)

In [1]:
import pandas as pd  # pandas truoc torch (Windows)
import numpy as np, json, math, collections, inspect, warnings, os, sys
import torch
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1"); os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
import qa_model as qm
import transformers; transformers.logging.set_verbosity_error()
from transformers import AutoTokenizer, AutoConfig, AutoModel
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 80)
md = lambda s: display(Markdown(s))
tok = AutoTokenizer.from_pretrained(qm.MODEL_NAME)
data = qm.make_splits()
print({k: len(v) for k, v in data.items()}, "| tokenizer:", qm.MODEL_NAME, "| fast:", tok.is_fast)

{'train': 20000, 'validation': 1000, 'test': 10570} | tokenizer: bert-base-cased | fast: True


## 1. Calculation Exercise 1 — Start / End (mục 6)

Context sau khi tokenize (đề cho): `[The, capital, of, France, is, Paris, .]`, question `What is the capital of France?`, answer `Paris`. Chỉ số **bắt đầu từ 0** *trong danh sách token của context* (đề: `start = end = 5`).
Sau đó đổi answer thành `capital of France` ⇒ `start = 1`, `end = 3` (**end là chỉ số của token cuối, bao gồm**, không phải exclusive).

In [2]:
ctx_tokens = ["The", "capital", "of", "France", "is", "Paris", "."]
def span_of(answer_words):
    n = len(answer_words)
    for i in range(len(ctx_tokens) - n + 1):
        if ctx_tokens[i:i + n] == answer_words: return i, i + n - 1
s1 = span_of(["Paris"]); s2 = span_of(["capital", "of", "France"])
assert s1 == (5, 5) and s2 == (1, 3)
md(f"- `Paris` → start = **{s1[0]}**, end = **{s1[1]}** (span 1 token; điều kiện `i ≤ j` thoả).\n"
   f"- `capital of France` → start = **{s2[0]}**, end = **{s2[1]}** (3 token: `capital`=1, `of`=2, `France`=3).")

# Cung vi tri nhin tu input that cua model: [CLS] question [SEP] context [SEP]
question = "What is the capital of France?"; context = "The capital of France is Paris."
enc = tok(question, context)
toks = tok.convert_ids_to_tokens(enc["input_ids"]); seq = enc.sequence_ids()
cs = seq.index(1)
df = pd.DataFrame({"idx in full sequence": range(len(toks)), "token": toks, "sequence_id": seq})
df["idx in context"] = [i - cs if s == 1 else None for i, s in enumerate(seq)]
display(df.T)
md(f"Trong chuỗi đầu vào đầy đủ của model, token context đầu tiên ở vị trí **{cs}** (sau `[CLS]` + {cs-2} token question + `[SEP]`) ⇒ `Paris` có index **{cs+5}** trong `input_ids` nhưng **5** trong danh sách token của context. "
   "Model dự đoán `start/end` trên **toàn chuỗi**, nên label phải là index toàn chuỗi (= index trong context + offset của question).")
assert toks[cs + 5] == "Paris"

- `Paris` → start = **5**, end = **5** (span 1 token; điều kiện `i ≤ j` thoả).
- `capital of France` → start = **1**, end = **3** (3 token: `capital`=1, `of`=2, `France`=3).

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
idx in full sequence,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
token,[CLS],What,is,the,capital,of,France,?,[SEP],The,capital,of,France,is,Paris,.,[SEP]
sequence_id,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN
idx in context,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,2.0,3.0,4.0,5.0,6.0,NaN


Trong chuỗi đầu vào đầy đủ của model, token context đầu tiên ở vị trí **9** (sau `[CLS]` + 7 token question + `[SEP]`) ⇒ `Paris` có index **14** trong `input_ids` nhưng **5** trong danh sách token của context. Model dự đoán `start/end` trên **toàn chuỗi**, nên label phải là index toàn chuỗi (= index trong context + offset của question).

## 2. Calculation Exercise 2 — Start loss & QA loss (mục 7, 8)

Model dự đoán phân phối `P_start` trên 7 vị trí; gold start = 5. $L_{start} = -\log P_{start}(5) = -\log 0.60$.
Đề không cho bảng `P_end`; ta lấy **cùng bảng** (answer 1 token ⇒ end = 5) rồi lấy thêm một bảng *end khác* để thấy hai loss có thể khác nhau.

In [3]:
pos = list(range(7))
P_start = np.array([0.02, 0.05, 0.03, 0.10, 0.15, 0.60, 0.05]); assert abs(P_start.sum() - 1) < 1e-12
P_end   = np.array([0.01, 0.04, 0.02, 0.08, 0.10, 0.70, 0.05]); assert abs(P_end.sum() - 1) < 1e-12   # vi du bang end (tu dat)
i_gold = j_gold = 5
L_start = -math.log(P_start[i_gold]); L_end = -math.log(P_end[j_gold]); L = (L_start + L_end) / 2
# kiem chung bang torch (cross-entropy tren logits = log p)
Lt = float(qm.qa_loss(torch.tensor(P_start).log()[None], torch.tensor(P_end).log()[None], torch.tensor([5]), torch.tensor([5])))
assert abs(L - Lt) < 1e-9
display(pd.DataFrame({"position": pos, "P_start": P_start, "P_end": P_end}).T)
md(f"- $L_{{start}} = -\\log 0.60 = $ **{L_start:.4f}** (nats)  — tương đương `{-math.log2(0.6):.4f}` bit.\n"
   f"- $L_{{end}} = -\\log 0.70 = $ **{L_end:.4f}** (bảng end tự đặt).\n"
   f"- $L = (L_{{start}} + L_{{end}})/2 = $ **{L:.4f}**  (khớp `qm.qa_loss` = {Lt:.4f}).\n"
   f"- Nếu model chỉ đặt 0.20 cho vị trí đúng (đoán gần như đều): $-\\log 0.20 = {-math.log(0.2):.4f}$; nếu 0.99: ${-math.log(0.99):.4f}$ — loss → 0 khi xác suất → 1; loss *không* bị chặn khi xác suất → 0.")

,0,1,2,3,4,5,6
position,0.00,1.00,2.00,3.00,4.00,5.0,6.00
P_start,0.02,0.05,0.03,0.10,0.15,0.6,0.05
P_end,0.01,0.04,0.02,0.08,0.10,0.7,0.05


- $L_{start} = -\log 0.60 = $ **0.5108** (nats)  — tương đương `0.7370` bit.
- $L_{end} = -\log 0.70 = $ **0.3567** (bảng end tự đặt).
- $L = (L_{start} + L_{end})/2 = $ **0.4338**  (khớp `qm.qa_loss` = 0.4338).
- Nếu model chỉ đặt 0.20 cho vị trí đúng (đoán gần như đều): $-\log 0.20 = 1.6094$; nếu 0.99: $0.0101$ — loss → 0 khi xác suất → 1; loss *không* bị chặn khi xác suất → 0.

### Vì sao cần **hai** prediction (start và end) thay vì một?

1. **Một span cần hai bậc tự do.** Span `(i, j)` gồm *vị trí* và *độ dài*. Chỉ dự đoán `start` không nói được span dài bao nhiêu: cùng `start = 1` (`capital`) có thể là `capital`, `capital of`, `capital of France` (bảng dưới). Chỉ dự đoán một vị trí giống *token classification một nhãn*, không mã hoá được ranh giới.
2. **Hai softmax thay cho $O(T^2)$ lớp.** Có $T(T+1)/2$ span hợp lệ; thay vì một classifier $T(T+1)/2$ lớp (không chia sẻ tham số giữa các span), ta phân tích $P(i,j)\approx P_{start}(i)\,P_{end}(j)$ (ràng buộc $i \le j$ và $j-i+1 \le L_{max}$ lúc suy luận): chỉ cần **2T** logit từ một `Linear(d, 2)`, mỗi token chỉ cần biết "tôi có phải biên trái / biên phải không".
3. **Mỗi biên có tín hiệu khác nhau.** Biên trái thường là một từ khoá rõ (chữ hoa, số); biên phải mơ hồ hơn (giữ hay bỏ giới từ, mệnh đề phụ) — hai head tách riêng cho phép phân tích riêng wrong-start vs wrong-end (mục 25).
4. **Khác NER/BIO:** NER gán nhãn *mỗi token* với nhiều span; QA chỉ có **một** span ⇒ softmax **trên các vị trí** (đúng một vị trí là start) chứ không phải softmax trên nhãn cho từng vị trí.

*Giá phải trả:* start/end độc lập có thể cho cặp không nhất quán (end < start, span quá dài) ⇒ phải giải mã có ràng buộc (`postprocess`: n-best, `end ≥ start`, độ dài ≤ 30).

In [4]:
# So span hop le va joint P(i,j) ~ P_start(i) P_end(j) voi rang buoc i <= j
T = len(P_start); n_spans = T * (T + 1) // 2
J = np.outer(P_start, P_end); J_valid = np.triu(J)               # i <= j
best = np.unravel_index(J_valid.argmax(), J.shape)
mass_invalid = np.tril(J, -1).sum()
display(pd.DataFrame(J_valid, index=[f"i={k}" for k in pos], columns=[f"j={k}" for k in pos]).round(4))
md(f"T = {T} token ⇒ **{n_spans}** span hợp lệ nhưng chỉ cần **{2*T}** logit; span tốt nhất theo `P_start·P_end` là {tuple(int(x) for x in best)} (xác suất tích {J_valid.max():.3f}). "
   f"Khối xác suất rơi vào cặp **không hợp lệ** (`j < i`) là {mass_invalid:.3f} — bị loại khi giải mã (đó là lý do phải có ràng buộc).")
# cung start=1, nhieu end khac nhau
for j in (1, 2, 3): print(f"start=1, end={j}: span =", " ".join(ctx_tokens[1:j+1]))

,j=0,j=1,j=2,j=3,j=4,j=5,j=6
i=0,0.0002,0.0008,0.0004,0.0016,0.002,0.014,0.0010
i=1,0.0000,0.0020,0.0010,0.0040,0.005,0.035,0.0025
i=2,0.0000,0.0000,0.0006,0.0024,0.003,0.021,0.0015
i=3,0.0000,0.0000,0.0000,0.0080,0.010,0.070,0.0050
i=4,0.0000,0.0000,0.0000,0.0000,0.015,0.105,0.0075
i=5,0.0000,0.0000,0.0000,0.0000,0.000,0.420,0.0300
i=6,0.0000,0.0000,0.0000,0.0000,0.000,0.000,0.0025


T = 7 token ⇒ **28** span hợp lệ nhưng chỉ cần **14** logit; span tốt nhất theo `P_start·P_end` là (5, 5) (xác suất tích 0.420). Khối xác suất rơi vào cặp **không hợp lệ** (`j < i`) là 0.229 — bị loại khi giải mã (đó là lý do phải có ràng buộc).

start=1, end=1: span = capital
start=1, end=2: span = capital of
start=1, end=3: span = capital of France


## 3. Tokenization của QA — `input_ids`, `attention_mask`, `token_type_ids` (mục 10)

Đầu vào có dạng `[CLS] question [SEP] context [SEP]`. **`token_type_ids` chỉ có nếu model dùng segment embedding** (BERT: `type_vocab_size = 2`). Đề dặn *không giả định mọi Transformer hiện đại đều dùng `token_type_ids`* ⇒ kiểm tra bằng **config / chữ ký `forward`** của model, không chỉ bằng đầu ra của tokenizer.

In [5]:
enc = tok(question, context, return_offsets_mapping=True)
rows = pd.DataFrame({"i": range(len(enc["input_ids"])), "token": tok.convert_ids_to_tokens(enc["input_ids"]), "input_ids": enc["input_ids"],
                     "attention_mask": enc["attention_mask"], "token_type_ids": enc["token_type_ids"], "sequence_id": enc.sequence_ids(),
                     "offset": enc["offset_mapping"]})
display(rows)
md(f"`token_type_ids = 0` cho `[CLS] question [SEP]` và `1` cho `context [SEP]` (segment A/B); `attention_mask = 1` cho mọi token thật (padding mới là 0). "
   f"`sequence_id` = 0 (question) / 1 (context) / None (special) — dùng để *chỉ nhận span trong context*.")
assert enc["token_type_ids"][:cs] == [0] * cs and set(enc["token_type_ids"][cs:]) == {1}

,i,token,input_ids,attention_mask,token_type_ids,sequence_id,offset
0,0,[CLS],101,1,0,NaN,"(0, 0)"
1,1,What,1327,1,0,0.0,"(0, 4)"
2,2,is,1110,1,0,0.0,"(5, 7)"
3,3,the,1103,1,0,0.0,"(8, 11)"
4,4,capital,2364,1,0,0.0,"(12, 19)"
5,5,of,1104,1,0,0.0,"(20, 22)"
6,6,France,1699,1,0,0.0,"(23, 29)"
7,7,?,136,1,0,0.0,"(29, 30)"
8,8,[SEP],102,1,0,NaN,"(0, 0)"
9,9,The,1109,1,1,1.0,"(0, 3)"


`token_type_ids = 0` cho `[CLS] question [SEP]` và `1` cho `context [SEP]` (segment A/B); `attention_mask = 1` cho mọi token thật (padding mới là 0). `sequence_id` = 0 (question) / 1 (context) / None (special) — dùng để *chỉ nhận span trong context*.

In [6]:
# Cac model khac nhau co dung token_type_ids khong? (kiem tra tokenizer VA model)
rows = []
for name in ["bert-base-cased", "distilbert-base-cased", "roberta-base", "distilroberta-base"]:
    try:
        t = AutoTokenizer.from_pretrained(name); cfg = AutoConfig.from_pretrained(name)
        e = t(question, context)
        m = AutoModel.from_config(cfg)
        accepts = "token_type_ids" in inspect.signature(m.forward).parameters
        rows.append({"model": name, "tokenizer returns token_type_ids": "token_type_ids" in e, "type_vocab_size": getattr(cfg, "type_vocab_size", None),
                     "model.forward accepts token_type_ids": accepts, "special tokens": t.decode(e["input_ids"][:2]) + " ... " + t.decode(e["input_ids"][-2:])})
    except Exception as ex:
        rows.append({"model": name, "tokenizer returns token_type_ids": f"(khong tai duoc: {type(ex).__name__})"})
display(pd.DataFrame(rows))
md("Kết luận: BERT dùng `token_type_ids`; RoBERTa không trả/không nhận (`type_vocab_size = 1`, hai segment ngăn bởi `</s></s>`); **DistilBERT** là ví dụ hay: tokenizer vẫn trả `token_type_ids` nhưng `forward` của model **không nhận** ⇒ nếu truyền vào sẽ lỗi. "
   "`qa_model.QAFeatures` chỉ đưa `token_type_ids` vào batch khi tokenizer trả về, và `QAModel.hidden_states` chỉ truyền khi có (xem `make_collate`/`_inputs`); với model không nhận thì cần bỏ cột này.")

,model,tokenizer returns token_type_ids,type_vocab_size,model.forward accepts token_type_ids,special tokens
0,bert-base-cased,True,2.0,True,[CLS] What ... . [SEP]
1,distilbert-base-cased,True,NaN,False,[CLS] What ... . [SEP]
2,roberta-base,False,1.0,True,<s>What ... .</s>
3,distilroberta-base,False,1.0,True,<s>What ... .</s>


Kết luận: BERT dùng `token_type_ids`; RoBERTa không trả/không nhận (`type_vocab_size = 1`, hai segment ngăn bởi `</s></s>`); **DistilBERT** là ví dụ hay: tokenizer vẫn trả `token_type_ids` nhưng `forward` của model **không nhận** ⇒ nếu truyền vào sẽ lỗi. `qa_model.QAFeatures` chỉ đưa `token_type_ids` vào batch khi tokenizer trả về, và `QAModel.hidden_states` chỉ truyền khi có (xem `make_collate`/`_inputs`); với model không nhận thì cần bỏ cột này.

## 4. Offset mapping & Calculation Exercise 3 — Character span (mục 11, 12)

SQuAD cho `answer_text` + `answer_start` theo **ký tự**; Transformer làm việc trên **vị trí token**. Cần phân biệt **character index / word index / token index**.

In [7]:
ctx = "The capital of France is Paris."
a_text = "Paris"; a_start = ctx.index(a_text); a_end_incl = a_start + len(a_text) - 1; a_end_excl = a_start + len(a_text)
assert ctx[a_start:a_end_excl] == a_text
md("**Calculation Exercise 3** — context `The capital of France is Paris.`\n\n"
   f"1. index ký tự đầu tiên của `Paris` = **{a_start}**\n2. index ký tự cuối = **{a_end_incl}** (nếu dùng biên phải *exclusive*: **{a_end_excl}**; dấu `.` ở index {a_end_excl})\n"
   f"3. answer span theo character = **[{a_start}, {a_end_excl})** ⇒ `ctx[{a_start}:{a_end_excl}] = '{ctx[a_start:a_end_excl]}'` (độ dài {a_end_excl-a_start})")
e = qm.tokenize_qa(tok, [question], [ctx], 384, 128)
seq = e.sequence_ids(0); offs = [o if seq[k] == 1 else (0, 0) for k, o in enumerate(e["offset_mapping"][0])]
sp = paris_sp = qm.align_span(offs, seq, a_start, a_end_excl)
t_ = tok.convert_ids_to_tokens(e["input_ids"][0])
md(f"4. sau tokenization, token tương ứng = `{t_[sp[0]]}` với offset `{offs[sp[0]]}` (⊇ span ký tự [{a_start}, {a_end_excl}))\n5. **start token = {sp[0]}**  6. **end token = {sp[1]}**  (index *trong chuỗi đầy đủ*; `{sp[0]-cs}` nếu chỉ đếm trong context).")
assert (a_start, a_end_incl) == (25, 29) and sp[0] == sp[1] == cs + 5
# char vs word vs token index
words = ctx.split(); w_idx = [i for i, w in enumerate(words) if "Paris" in w][0]
display(pd.DataFrame({"đơn vị": ["character index", "word index (split khoảng trắng)", "token index (trong context)", "token index (toàn chuỗi)"],
                      "giá trị của `Paris`": [a_start, w_idx, sp[0] - cs, sp[0]],
                      "lưu ý": ["đầu ký tự; SQuAD answer_start dùng đơn vị này", f"từ là `{words[w_idx]}` (dính dấu `.`) → khác token `Paris`", "WordPiece tách `.` thành token riêng", "label cho start/end positions"]}))

**Calculation Exercise 3** — context `The capital of France is Paris.`

1. index ký tự đầu tiên của `Paris` = **25**
2. index ký tự cuối = **29** (nếu dùng biên phải *exclusive*: **30**; dấu `.` ở index 30)
3. answer span theo character = **[25, 30)** ⇒ `ctx[25:30] = 'Paris'` (độ dài 5)

4. sau tokenization, token tương ứng = `Paris` với offset `(25, 30)` (⊇ span ký tự [25, 30))
5. **start token = 14**  6. **end token = 14**  (index *trong chuỗi đầy đủ*; `5` nếu chỉ đếm trong context).

,đơn vị,giá trị của `Paris`,lưu ý
0,character index,25,đầu ký tự; SQuAD answer_start dùng đơn vị này
1,word index (split khoảng trắng),5,từ là `Paris.` (dính dấu `.`) → khác token `Paris`
2,token index (trong context),5,WordPiece tách `.` thành token riêng
3,token index (toàn chuỗi),14,label cho start/end positions


### Pseudo-code của đề vs `align_span`

Đề đưa pseudo-code (`start <= answer_start < end` → start_token; `start < answer_end <= end` → end_token). Cài **đúng nguyên văn** (duyệt mọi token của `offset_mapping` *thô* do tokenizer trả về — gồm cả token của question, mà offset của chúng là vị trí **trong chuỗi question**, không phải trong context) rồi đối chiếu với `qm.align_span` (chỉ xét token context, có nhánh *answer ngoài window*) trên toàn bộ window của tập train (max_len = 384, stride = 128). Phải kiểm tra các trường hợp đề nêu: *answer bắt đầu giữa token*, *kết thúc giữa token*, *bị truncation*, *nằm ngoài window*.

In [8]:
def align_pseudocode(offsets, answer_start, answer_end):
    # NGUYEN VAN pseudo-code cua de (muc 16): duyet MOI token cua offset_mapping THO (ke ca question/special), khong kiem tra window
    start_token = end_token = None
    for token_idx, (start, end) in enumerate(offsets):
        if start <= answer_start < end: start_token = token_idx
        if start < answer_end <= end:   end_token = token_idx
    return start_token, end_token

tr = data["train"]
A = [(r["answers"]["answer_start"][0], r["answers"]["answer_start"][0] + len(r["answers"]["text"][0])) for r in tr]
enc384 = qm.tokenize_qa(tok, [r["question"] for r in tr], [r["context"] for r in tr], 384, 128)
cmp = collections.Counter(); examples = collections.defaultdict(list)
for i in range(len(enc384["input_ids"])):
    ex = enc384["overflow_to_sample_mapping"][i]; sq = enc384.sequence_ids(i)
    of = [o if sq[k] == 1 else (0, 0) for k, o in enumerate(enc384["offset_mapping"][i])]
    kind = qm.alignment_kind(of, sq, *A[ex]); naive = align_pseudocode(enc384["offset_mapping"][i], *A[ex]); mine = qm.align_span(of, sq, *A[ex])
    cmp[kind] += 1
    if kind == "outside_window":
        status = "pseudo-code tra None (dung)" if naive[0] is None or naive[1] is None else "pseudo-code tra span SAI (khong None) - label rac"
        cmp["  -> " + status] += 1
    else:
        cmp["  -> pseudo-code == align_span" if naive == mine else "  -> pseudo-code KHAC align_span"] += 1
    if len(examples[kind]) < 3: examples[kind].append((i, ex))
display(pd.Series(cmp, name="so window").to_frame())
bad = sum(v for k, v in cmp.items() if "SAI" in k or "KHAC" in k)
md(f"Trên {sum(v for k, v in cmp.items() if not k.startswith('  ')):,} window, pseudo-code thô cho **{bad}** label khác/sai so với `align_span` "
   f"({cmp.get('  -> pseudo-code tra span SAI (khong None) - label rac', 0)} trường hợp *answer ngoài window* mà pseudo-code vẫn trả một cặp token — thường là khớp nhầm với token của question vì offset question là vị trí trong question; "
   f"{cmp.get('  -> pseudo-code KHAC align_span', 0)} trường hợp khác ở window chứa answer). `align_span` giới hạn vào token context (`sequence_id == 1`), kiểm tra answer nằm trọn trong khoảng context của window trước khi gán nhãn, và gán `[CLS]` khi không.")

,so window
exact,19985
-> pseudo-code == align_span,20047
outside_window,187
-> pseudo-code tra None (dung),181
start_mid_token,23
end_mid_token,36
-> pseudo-code tra span SAI (khong None) - label rac,6
both_mid_token,3


Trên 20,234 window, pseudo-code thô cho **6** label khác/sai so với `align_span` (6 trường hợp *answer ngoài window* mà pseudo-code vẫn trả một cặp token — thường là khớp nhầm với token của question vì offset question là vị trí trong question; 0 trường hợp khác ở window chứa answer). `align_span` giới hạn vào token context (`sequence_id == 1`), kiểm tra answer nằm trọn trong khoảng context của window trước khi gán nhãn, và gán `[CLS]` khi không.

In [9]:
def show_case(i, ex, enc_, max_show=6):
    sq = enc_.sequence_ids(i); of = [o if sq[k] == 1 else (0, 0) for k, o in enumerate(enc_["offset_mapping"][i])]
    a_s, a_e = A[ex]; r = tr[ex]; sp = qm.align_span(of, sq, a_s, a_e); toks_ = tok.convert_ids_to_tokens(enc_["input_ids"][i])
    out = {"question": r["question"], "gold answer": r["answers"]["text"][0], "char span": (a_s, a_e), "kind": qm.alignment_kind(of, sq, a_s, a_e)}
    if sp:
        out["start/end token"] = sp; out["tokens"] = " ".join(toks_[sp[0]:sp[1] + 1])
        out["token offsets"] = (of[sp[0]][0], of[sp[1]][1]); out["text recovered from offsets"] = r["context"][of[sp[0]][0]:of[sp[1]][1]]
    return out
for kind in ("start_mid_token", "end_mid_token", "both_mid_token"):
    md(f"**{kind}**")
    display(pd.DataFrame([show_case(i, ex, enc384) for i, ex in examples[kind]]))
md("Khi answer bắt đầu/kết thúc **giữa** một token (vd. gold `2` nằm trong token `72`; gold `No` là phần đầu của token `Nonetheless`; gold `60 w` kết thúc giữa `wa…`), token chứa ký tự đó được chọn nên **toàn bộ token** nằm trong span ⇒ văn bản khôi phục từ offset dài hơn gold vài ký tự. "
   "Đây là nhãn nhiễu của SQuAD (answer cắt giữa từ), rất hiếm (xem số đếm ở trên) và ảnh hưởng EM rất nhỏ (xem *oracle EM* ở mục 7).")

**start_mid_token**

,question,gold answer,char span,kind,start/end token,tokens,token offsets,text recovered from offsets
0,How much rainfall in inches per hour can the torch stay lit under?,2,"(145, 146)",start_mid_token,"(47, 47)",72,"(144, 146)",72
1,What makes up the sum of relations to an entity?,rld around it,"(176, 189)",start_mid_token,"(46, 48)",world around it,"(174, 189)",world around it
2,Who was Edward j Gurney,"968 Edward J. Gurney, also a white conservative, was elected as the state's ...","(639, 762)",start_mid_token,"(137, 162)","1968 Edward J . G ##urney , also a white conservative , was elected as the s...","(638, 762)","1968 Edward J. Gurney, also a white conservative, was elected as the state's..."


**end_mid_token**

,question,gold answer,char span,kind,start/end token,tokens,token offsets,text recovered from offsets
0,Is the age that peope are allowed to marry universally decided upon?,No,"(593, 595)",end_mid_token,"(129, 129)",Nonetheless,"(593, 604)",Nonetheless
1,How many national newspapers out of the three are from New York?,2,"(10, 11)",end_mid_token,"(17, 17)",200,"(10, 13)",200
2,Huge much light does a 13 W compact fluorescent lamp produce equivalent to a...,60 w,"(870, 874)",end_mid_token,"(195, 196)",60 wa,"(870, 875)",60 wa


**both_mid_token**

,question,gold answer,char span,kind,start/end token,tokens,token offsets,text recovered from offsets
0,How many primary schools does the island have?,3,"(288, 289)",both_mid_token,"(74, 74)",230,"(287, 290)",230
1,Each Brigade contains how many regiments?,one,"(33, 36)",both_mid_token,"(14, 14)",component,"(29, 38)",component
2,v,v,"(108, 109)",both_mid_token,"(30, 30)",River,"(106, 111)",River


Khi answer bắt đầu/kết thúc **giữa** một token (vd. gold `2` nằm trong token `72`; gold `No` là phần đầu của token `Nonetheless`; gold `60 w` kết thúc giữa `wa…`), token chứa ký tự đó được chọn nên **toàn bộ token** nằm trong span ⇒ văn bản khôi phục từ offset dài hơn gold vài ký tự. Đây là nhãn nhiễu của SQuAD (answer cắt giữa từ), rất hiếm (xem số đếm ở trên) và ảnh hưởng EM rất nhỏ (xem *oracle EM* ở mục 7).

In [10]:
# Answer nam ngoai window / bi truncation cat (max_len nho de de thay)
tr_small = tr[:2000]
A_small = A[:2000]
e128 = qm.tokenize_qa(tok, [r["question"] for r in tr_small], [r["context"] for r in tr_small], 128, 42)
shown = 0
for i in range(len(e128["input_ids"])):
    ex = e128["overflow_to_sample_mapping"][i]; sq = e128.sequence_ids(i)
    of = [o if sq[k] == 1 else (0, 0) for k, o in enumerate(e128["offset_mapping"][i])]
    cs_i, ce_i = qm.context_token_range(sq)
    a_s, a_e = A_small[ex]
    kind = qm.alignment_kind(of, sq, a_s, a_e)
    cut = of[cs_i][0] <= a_s < of[ce_i][1] < a_e         # answer bat dau trong window nhung bi cat o bien phai
    if kind == "outside_window" and cut and shown < 3:
        shown += 1; r = tr_small[ex]
        print(f"[window {i} of example {ex}] question: {r['question']!r}\n  gold answer {r['answers']['text'][0]!r} @ chars {A_small[ex]}; window covers context chars [{of[cs_i][0]}, {of[ce_i][1]}) "
              f"-> answer BI CAT o bien phai => label = [CLS] (0, 0) trong window nay; example nay con {sum(1 for x in e128['overflow_to_sample_mapping'] if x == ex)} window")

[window 295 of example 137] question: 'What typically happened aboard the ship on wheels?'
  gold answer 'the marriage of a man and woman be consummated' @ chars (524, 570); window covers context chars [0, 555) -> answer BI CAT o bien phai => label = [CLS] (0, 0) trong window nay; example nay con 2 window
[window 460 of example 214] question: 'Who had done better, the Prussians in Europe or the Brits in North America?'
  gold answer 'successful for the Prussian-led forces on the continent, in contrast to disappointing British campaigns in North America.' @ chars (523, 644); window covers context chars [0, 533) -> answer BI CAT o bien phai => label = [CLS] (0, 0) trong window nay; example nay con 2 window
[window 769 of example 350] question: 'Who did Lockheed sign an agreement with to seek out more minority workers?'
  gold answer 'Vice President Johnson' @ chars (530, 552); window covers context chars [0, 534) -> answer BI CAT o bien phai => label = [CLS] (0, 0) trong window nay; exam

## 5. Truncation và sliding window (mục 13, 15)

`tokenizer(question, context, truncation="only_second", max_length, stride, return_overflowing_tokens=True, return_offsets_mapping=True)`: **chỉ cắt context** (question giữ nguyên); nếu context dài hơn, trả về *nhiều window* liên tiếp, hai window kề nhau **chồng nhau `stride` token**; `overflow_to_sample_mapping[k]` cho biết window `k` thuộc example nào. Dưới đây chọn một example có context dài và in các tensor đề yêu cầu.

In [11]:
lens = [len(tok(r["question"], r["context"])["input_ids"]) for r in data["train"][:3000]]
long_i = int(np.argsort(lens)[-1]); rlong = tr[long_i]
ML, ST = 128, 42                      # max_length nho de nhin ro nhieu window
encL = qm.tokenize_qa(tok, [rlong["question"]], [rlong["context"]], ML, ST)
n_w = len(encL["input_ids"])
print(f"question: {rlong['question']!r}\ncontext: {len(rlong['context'])} ky tu, {lens[long_i]} token (khong cat)  -> max_length={ML}, stride={ST}: {n_w} window")
print("keys:", list(encL.keys()))
print("overflow_to_sample_mapping:", encL["overflow_to_sample_mapping"])
rows = []
a_s0 = rlong["answers"]["answer_start"][0]; a_e0 = a_s0 + len(rlong["answers"]["text"][0])
prev_ids = None
for k in range(n_w):
    sq = encL.sequence_ids(k); cs_k, ce_k = qm.context_token_range(sq); of = encL["offset_mapping"][k]
    ids_ctx = [t for t, s in zip(encL["input_ids"][k], sq) if s == 1]
    sp = qm.align_span([o if sq[j] == 1 else (0, 0) for j, o in enumerate(of)], sq, a_s0, a_e0)
    rows.append({"window": k, "len(input_ids)": len(encL["input_ids"][k]), "sum(attention_mask)": sum(encL["attention_mask"][k]), "context tokens": len(ids_ctx),
                 "context chars": (of[cs_k][0], of[ce_k][1]), "overlap w/ prev (tokens)": (0 if prev_ids is None else next((n for n in range(min(len(prev_ids), len(ids_ctx)), 0, -1) if prev_ids[-n:] == ids_ctx[:n]), 0)),
                 "answer in window": sp is not None, "(start,end) label": sp if sp else "(0,0) = [CLS]"})
    prev_ids = ids_ctx
display(pd.DataFrame(rows))
md(f"- `input_ids` mỗi window ≤ {ML}; `attention_mask` toàn 1 (chưa padding); `offset_mapping` là ký tự (start, end) trong **context gốc** (question/special = `(0,0)`) nên các window sau vẫn trỏ về cùng chuỗi `context` ⇒ char span của gold không đổi giữa các window.\n"
   f"- Hai window kề nhau chồng đúng `stride = {ST}` token (cột *overlap*): token cuối window k lặp lại đầu window k+1, nhờ vậy một answer ngắn không bị cắt đôi ở **mọi** window.\n"
   f"- Gold answer `{rlong['answers']['text'][0]!r}` nằm trong {sum(1 for r_ in rows if r_['answer in window'])} window; ở các window còn lại label là `[CLS] = (0,0)` (answer ngoài window).")
assert all(r_["answer in window"] or r_["(start,end) label"] == "(0,0) = [CLS]" for r_ in rows)

question: "How many copies did Whitesnake's 1987 self title album sell?"
context: 2414 ky tu, 610 token (khong cat)  -> max_length=128, stride=42: 9 window
keys: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'overflow_to_sample_mapping']
overflow_to_sample_mapping: [0, 0, 0, 0, 0, 0, 0, 0, 0]


,window,len(input_ids),sum(attention_mask),context tokens,context chars,overlap w/ prev (tokens),answer in window,"(start,end) label"
0,0,128,128,109,"(0, 462)",0,True,"(43, 44)"
1,1,128,128,109,"(282, 739)",42,False,"(0,0) = [CLS]"
2,2,128,128,109,"(550, 1009)",42,False,"(0,0) = [CLS]"
3,3,128,128,109,"(838, 1256)",42,False,"(0,0) = [CLS]"
4,4,128,128,109,"(1089, 1568)",42,False,"(0,0) = [CLS]"
5,5,128,128,109,"(1400, 1789)",42,False,"(0,0) = [CLS]"
6,6,128,128,109,"(1631, 2056)",42,False,"(0,0) = [CLS]"
7,7,128,128,109,"(1898, 2358)",42,False,"(0,0) = [CLS]"
8,8,74,74,55,"(2190, 2414)",42,False,"(0,0) = [CLS]"


- `input_ids` mỗi window ≤ 128; `attention_mask` toàn 1 (chưa padding); `offset_mapping` là ký tự (start, end) trong **context gốc** (question/special = `(0,0)`) nên các window sau vẫn trỏ về cùng chuỗi `context` ⇒ char span của gold không đổi giữa các window.
- Hai window kề nhau chồng đúng `stride = 42` token (cột *overlap*): token cuối window k lặp lại đầu window k+1, nhờ vậy một answer ngắn không bị cắt đôi ở **mọi** window.
- Gold answer `'17 million'` nằm trong 1 window; ở các window còn lại label là `[CLS] = (0,0)` (answer ngoài window).

In [12]:
# Cat don gian (khong stride) co lam mat answer khong?  So sanh: truncation chi lay window dau  vs  sliding window
res = []
sub = data["train"]
for L in (128, 256, 384):
    st = qm.default_stride(L)
    f_sw = qm.QAFeatures(tok, [r["question"] for r in sub], [r["context"] for r in sub], L, st, A)
    e_tr = tok([r["question"].strip() for r in sub], [r["context"] for r in sub], truncation="only_second", max_length=L, return_offsets_mapping=True)
    lost = 0
    for k in range(len(sub)):
        sq = e_tr.sequence_ids(k); of = [o if sq[j] == 1 else (0, 0) for j, o in enumerate(e_tr["offset_mapping"][k])]
        lost += qm.align_span(of, sq, *A[k]) is None
    s = f_sw.stats()
    res.append({"max_length": L, "stride": st, "windows / example": round(s["windows_per_example"], 3), "features": s["n_features"],
                "answer lost (truncate only)": f"{lost} ({lost/len(sub):.2%})", "answer lost (sliding window)": f"{s['n_examples_answer_lost']} ({s['n_examples_answer_lost']/len(sub):.2%})",
                "windows w/o answer ([CLS] label)": f"{s['n_no_answer_windows']} ({s['n_no_answer_windows']/s['n_features']:.2%})", "mean tokens/window": round(s["mean_len"], 1)})
display(pd.DataFrame(res))
r128, r384 = res[0], res[2]
md(f"Cắt đơn giản (`only_second` không `stride`) **mất answer**: {r128['answer lost (truncate only)']} example ở max_length 128 và {r384['answer lost (truncate only)']} ở 384. Sliding window đưa tỉ lệ này về **0** vì mỗi vị trí context nằm trong ít nhất một window; cái giá là nhiều feature hơn "
   f"({r128['features']:,} ở 128 so với {r384['features']:,} ở 384), và ở 128 có {r128['windows w/o answer ([CLS] label)']} window **không chứa answer** (label `[CLS]`) — model phải học 'answer không ở đây' cho các window đó. "
   "Ở lúc suy luận ta lấy span có điểm cao nhất qua *tất cả* window của một example (`qm.postprocess`).")

,max_length,stride,windows / example,features,answer lost (truncate only),answer lost (sliding window),windows w/o answer ([CLS] label),mean tokens/window
0,128,42,2.198,43952,4105 (20.52%),0 (0.00%),19912 (45.30%),111.5
1,256,85,1.121,22420,280 (1.40%),0 (0.00%),1945 (8.68%),167.4
2,384,128,1.012,20234,20 (0.10%),0 (0.00%),187 (0.92%),175.0


Cắt đơn giản (`only_second` không `stride`) **mất answer**: 4105 (20.52%) example ở max_length 128 và 20 (0.10%) ở 384. Sliding window đưa tỉ lệ này về **0** vì mỗi vị trí context nằm trong ít nhất một window; cái giá là nhiều feature hơn (43,952 ở 128 so với 20,234 ở 384), và ở 128 có 19912 (45.30%) window **không chứa answer** (label `[CLS]`) — model phải học 'answer không ở đây' cho các window đó. Ở lúc suy luận ta lấy span có điểm cao nhất qua *tất cả* window của một example (`qm.postprocess`).

## 6. Shape — từ `input_ids` đến `start/end logits`

$B = 8,\; T = 384,\; d = 768$ ⇒ $H \in \mathbb{R}^{B\times T\times d}$; head `Linear(d, 2)` ⇒ $Z \in \mathbb{R}^{B\times T\times 2}$; tách thành `start_logits`, `end_logits` $\in \mathbb{R}^{B\times T}$; `softmax` **theo chiều T** (phân phối trên vị trí); loss `CrossEntropy(logits[B,T], target[B])`.

In [13]:
B, T = 8, 384
model = qm.QAModel().eval(); d = model.encoder.config.hidden_size
ids = torch.randint(1000, 5000, (B, T)); am = torch.ones(B, T, dtype=torch.long); tt = torch.zeros(B, T, dtype=torch.long)
with torch.no_grad():
    H = model.hidden_states(ids, am, tt); s_, e_ = model(ids, am, tt)
    Z = model.qa_outputs(H)
assert H.shape == (B, T, d) and Z.shape == (B, T, 2) and s_.shape == e_.shape == (B, T)
p_start = torch.softmax(s_, -1); assert torch.allclose(p_start.sum(-1), torch.ones(B), atol=1e-5)
n_head = sum(p.numel() for p in model.qa_outputs.parameters()); n_all, _ = qm.count_parameters(model)
display(pd.DataFrame([["input_ids", tuple(ids.shape)], ["attention_mask", tuple(am.shape)], ["token_type_ids", tuple(tt.shape)], ["H = encoder(x)", tuple(H.shape)],
                      ["Z = Linear(d,2)(H)", tuple(Z.shape)], ["start_logits / end_logits", tuple(s_.shape)], ["softmax over T (sum=1)", tuple(p_start.shape)],
                      ["start_positions / end_positions", (B,)]], columns=["tensor", "shape"]))
md(f"Head chỉ có **{n_head:,}** tham số ({d}×2 + 2) trên tổng **{n_all:,}** của model ⇒ gần như toàn bộ năng lực nằm ở backbone pretrained; head mới chỉ học *đọc* span từ biểu diễn ngữ cảnh. "
   f"(Classification LAB 08: `Linear(d, C)` trên `h[CLS]`; NER LAB 09: `Linear(d, 9)` trên mọi token; QA: `Linear(d, 2)` trên mọi token, nhưng softmax **qua T**.)")

,tensor,shape
0,input_ids,"(8, 384)"
1,attention_mask,"(8, 384)"
2,token_type_ids,"(8, 384)"
3,H = encoder(x),"(8, 384, 768)"
4,"Z = Linear(d,2)(H)","(8, 384, 2)"
5,start_logits / end_logits,"(8, 384)"
6,softmax over T (sum=1),"(8, 384)"
7,start_positions / end_positions,"(8,)"


Head chỉ có **1,538** tham số (768×2 + 2) trên tổng **107,721,218** của model ⇒ gần như toàn bộ năng lực nằm ở backbone pretrained; head mới chỉ học *đọc* span từ biểu diễn ngữ cảnh. (Classification LAB 08: `Linear(d, C)` trên `h[CLS]`; NER LAB 09: `Linear(d, 9)` trên mọi token; QA: `Linear(d, 2)` trên mọi token, nhưng softmax **qua T**.)

## 7. Phân tích độ dài và độ chính xác của căn chỉnh (hình `tokenization_analysis.png`)

- Phân bố độ dài `question + context` (token) so với `max_length` 128/256/384 ⇒ vì sao 128 gây nhiều window.
- **Oracle EM / F1**: nếu model đoán **chính xác** token span gold (từ `align_span`) rồi khôi phục text bằng offset, EM/F1 trên dev là bao nhiêu? Đây là **trần** do tokenization/căn chỉnh (không phải do model).

In [14]:
test = data["test"]
Lq = [len(tok(r["question"].strip(), r["context"])["input_ids"]) for r in test]
Lt_ = [len(tok(r["question"].strip(), r["context"])["input_ids"]) for r in data["train"]]
cover = {L: float(np.mean(np.array(Lq) <= L)) for L in (128, 256, 384, 512)}
# oracle: cat va khoi phuc text tu token span gold (cua gold answer dau tien)
qs, cs_ = [r["question"] for r in test], [r["context"] for r in test]
ft = qm.QAFeatures(tok, qs, cs_, 384, 128, [(r["answers"]["answer_start"][0], r["answers"]["answer_start"][0] + len(r["answers"]["text"][0])) for r in test])
orc = [None] * len(test)
for f in ft.feats:
    if f["start_position"] > 0 and orc[f["example_idx"]] is None:
        ex = f["example_idx"]; orc[ex] = cs_[ex][f["offsets"][f["start_position"]][0]:f["offsets"][f["end_position"]][1]]
orc = [o if o is not None else "" for o in orc]
golds = [r["answers"]["text"] for r in test]
m_or = qm.evaluate_predictions(orc, golds)
md(f"- Dev (test của lab): {len(test):,} câu; **{cover[384]:.1%}** có `question+context ≤ 384` token (≤256: {cover[256]:.1%}; ≤128: {cover[128]:.1%}); median {int(np.median(Lq))}, p95 {int(np.percentile(Lq,95))}, max {max(Lq)}.\n"
   f"- **Oracle** (đoán đúng token span gold → khôi phục text từ offset, max_len 384): EM = **{m_or['em']:.4f}**, F1 = **{m_or['f1']:.4f}**, n = {m_or['n']:,}. "
   f"Trần này < 1 chỉ vì một số answer bắt đầu/kết thúc giữa token (xem mục 4) và vì khoảng trắng/dấu câu khi khôi phục; nó nhỏ so với khoảng cách model–gold ở `qa_finetuning.ipynb`.")
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].hist(Lt_, bins=60, color="#4C78A8", alpha=.85, label="train subset"); ax[0].hist(Lq, bins=60, color="#F58518", alpha=.5, label="dev")
for L, c in zip((128, 256, 384), ("#d62728", "#2ca02c", "#9467bd")): ax[0].axvline(L, color=c, ls="--", label=f"max_length={L}")
ax[0].set_xlabel("#token (question + context)"); ax[0].set_ylabel("#example"); ax[0].legend(fontsize=7); ax[0].set_title("Phan bo do dai")
rs = pd.DataFrame(res)
ax[1].bar([str(x) for x in rs["max_length"]], rs["windows / example"], color="#4C78A8"); ax[1].set_title("Windows / example (sliding window)"); ax[1].set_xlabel("max_length")
for i, v in enumerate(rs["windows / example"]): ax[1].text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
lost_trunc = [float(x.split("(")[1].strip("%)")) for x in rs["answer lost (truncate only)"]]
ax[2].bar([str(x) for x in rs["max_length"]], lost_trunc, color="#E45756", label="cat don gian (khong stride)"); ax[2].bar([str(x) for x in rs["max_length"]], [0, 0, 0], color="#54A24B", label="sliding window")
ax[2].set_ylabel("% example mat answer"); ax[2].set_xlabel("max_length"); ax[2].set_title("Answer bi mat do truncation"); ax[2].legend(fontsize=7)
plt.tight_layout(); plt.savefig("tokenization_analysis.png", dpi=130); plt.close()
display(Markdown("![tokenization_analysis](tokenization_analysis.png)"))
json.dump({"oracle": m_or, "coverage": cover, "median_len": int(np.median(Lq)), "p95_len": int(np.percentile(Lq, 95))}, open("data_cache/tokenization_stats.json", "w"))

- Dev (test của lab): 10,570 câu; **97.9%** có `question+context ≤ 384` token (≤256: 87.7%; ≤128: 19.0%); median 167, p95 308, max 833.
- **Oracle** (đoán đúng token span gold → khôi phục text từ offset, max_len 384): EM = **0.9978**, F1 = **0.9983**, n = 10,570. Trần này < 1 chỉ vì một số answer bắt đầu/kết thúc giữa token (xem mục 4) và vì khoảng trắng/dấu câu khi khôi phục; nó nhỏ so với khoảng cách model–gold ở `qa_finetuning.ipynb`.

![tokenization_analysis](tokenization_analysis.png)